In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import os
from typing import Tuple, Dict, Any

In [4]:
    class GoldDataPreprocessor:
        def __init__(self):
            self.scalers = {}
            self.data_dir = "data"
            self.models_dir = "saved_models"
            os.makedirs(self.models_dir, exist_ok=True)
        
        def create_technical_indicators(self, data: pd.DataFrame) -> pd.DataFrame:
            """Create technical indicators for gold price prediction"""
            df = data.copy()
            
            # Moving averages
            df['gold_ma_5'] = df['gold_close'].rolling(window=5).mean()
            df['gold_ma_10'] = df['gold_close'].rolling(window=10).mean()
            df['gold_ma_20'] = df['gold_close'].rolling(window=20).mean()
            df['gold_ma_50'] = df['gold_close'].rolling(window=50).mean()
            
            # Relative Strength Index (RSI)
            delta = df['gold_close'].diff()
            gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
            loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
            rs = gain / loss
            df['gold_rsi'] = 100 - (100 / (1 + rs))
            
            # Bollinger Bands
            df['gold_bb_middle'] = df['gold_close'].rolling(window=20).mean()
            bb_std = df['gold_close'].rolling(window=20).std()
            df['gold_bb_upper'] = df['gold_bb_middle'] + (bb_std * 2)
            df['gold_bb_lower'] = df['gold_bb_middle'] - (bb_std * 2)
            df['gold_bb_width'] = df['gold_bb_upper'] - df['gold_bb_lower']
            
            # Price momentum
            df['gold_momentum_1'] = df['gold_close'].pct_change(1)
            df['gold_momentum_5'] = df['gold_close'].pct_change(5)
            df['gold_momentum_10'] = df['gold_close'].pct_change(10)
            
            # Volatility (rolling standard deviation)
            df['gold_volatility_5'] = df['gold_close'].rolling(window=5).std()
            df['gold_volatility_20'] = df['gold_close'].rolling(window=20).std()
            
            # High-Low spread
            df['gold_hl_spread'] = (df['gold_high'] - df['gold_low']) / df['gold_close']
            
            # Volume indicators
            df['gold_volume_ma'] = df['gold_volume'].rolling(window=10).mean()
            df['gold_volume_ratio'] = df['gold_volume'] / df['gold_volume_ma']
            
            return df
        
        def create_lags(self, data: pd.DataFrame, target_col: str, lags: list) -> pd.DataFrame:
            """Create lagged features"""
            df = data.copy()
            
            for lag in lags:
                df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag)
            
            return df
        
        def create_rolling_features(self, data: pd.DataFrame) -> pd.DataFrame:
            """Create rolling statistical features"""
            df = data.copy()
            
            # For external indicators, create rolling features
            external_cols = [col for col in df.columns if col.endswith('_close') and not col.startswith('gold')]
            
            for col in external_cols:
                if col in df.columns:
                    # Rolling means
                    df[f'{col}_ma_5'] = df[col].rolling(window=5).mean()
                    df[f'{col}_ma_10'] = df[col].rolling(window=10).mean()
                    
                    # Rolling volatility
                    df[f'{col}_vol_5'] = df[col].rolling(window=5).std()
                    
                    # Momentum
                    df[f'{col}_momentum'] = df[col].pct_change(1)
            
            return df
        
        def prepare_features(self, data: pd.DataFrame) -> pd.DataFrame:
            """Complete feature engineering pipeline"""
            print("Creating technical indicators...")
            df = self.create_technical_indicators(data)
            
            print("Creating rolling features...")
            df = self.create_rolling_features(df)
            
            print("Creating lag features...")
            df = self.create_lags(df, 'gold_close', [1, 2, 3, 5, 7])
            
            # Create target variable (next day's gold price)
            df['target'] = df['gold_close'].shift(-1)
            
            # Drop rows with NaN values
            df = df.dropna()
            
            return df
        
        def scale_features(self, X_train: pd.DataFrame, X_test: pd.DataFrame, 
                        y_train: pd.Series, y_test: pd.Series) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
            """Scale features and target variables"""
            
            # Scale features
            feature_scaler = StandardScaler()
            X_train_scaled = feature_scaler.fit_transform(X_train)
            X_test_scaled = feature_scaler.transform(X_test)
            
            # Scale target
            target_scaler = StandardScaler()
            y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()
            y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1)).ravel()
            
            # Save scalers
            self.scalers['feature_scaler'] = feature_scaler
            self.scalers['target_scaler'] = target_scaler
            
            joblib.dump(feature_scaler, os.path.join(self.models_dir, 'feature_scaler.pkl'))
            joblib.dump(target_scaler, os.path.join(self.models_dir, 'target_scaler.pkl'))
            
            return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled
        
        def prepare_lstm_data(self, X: np.ndarray, y: np.ndarray, sequence_length: int = 60) -> Tuple[np.ndarray, np.ndarray]:
            """Prepare data for LSTM model"""
            X_lstm, y_lstm = [], []
            
            for i in range(sequence_length, len(X)):
                X_lstm.append(X[i-sequence_length:i])
                y_lstm.append(y[i])
            
            return np.array(X_lstm), np.array(y_lstm)
        
        def split_and_prepare_data(self, data: pd.DataFrame, test_size: float = 0.2) -> Dict[str, Any]:
            """Complete data preparation pipeline"""
            print("Preparing features...")
            processed_data = self.prepare_features(data)
            
            # Separate features and target
            feature_cols = [col for col in processed_data.columns if col != 'target']
            X = processed_data[feature_cols]
            y = processed_data['target']

            X = X.select_dtypes(include=[np.number])
            
            print(f"Dataset shape: {X.shape}")
            print(f"Features: {len(feature_cols)}")
            
            # Time series split (no shuffling)
            split_index = int(len(X) * (1 - test_size))
            X_train, X_test = X[:split_index], X[split_index:]
            y_train, y_test = y[:split_index], y[split_index:]
            
            # Scale data
            X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled = self.scale_features(
                X_train, X_test, y_train, y_test
            )
            
            # Prepare LSTM data
            X_train_lstm, y_train_lstm = self.prepare_lstm_data(X_train_scaled, y_train_scaled)
            X_test_lstm, y_test_lstm = self.prepare_lstm_data(X_test_scaled, y_test_scaled)
            
            return {
                'X_train': X_train_scaled,
                'X_test': X_test_scaled,
                'y_train': y_train_scaled,
                'y_test': y_test_scaled,
                'X_train_lstm': X_train_lstm,
                'X_test_lstm': X_test_lstm,
                'y_train_lstm': y_train_lstm,
                'y_test_lstm': y_test_lstm,
                'feature_names': feature_cols,
                'original_test_dates': X_test.index,
                'y_test_original': y_test
            }


In [8]:
def main():
    
    path = 'raw_gold_data.csv'
    combined_data = pd.read_csv(path, parse_dates=['Date'])
    print("Combined dataset shape:", combined_data.shape)

    # 2. Preprocess
    preprocessor = GoldDataPreprocessor()
    data_dict = preprocessor.split_and_prepare_data(combined_data, test_size=0.2)

    np.savez(
        "preprocessed_gold_data.npz",
        X_train=data_dict["X_train"],
        y_train=data_dict["y_train"],
        X_test=data_dict["X_test"],
        y_test=data_dict["y_test"],
        X_train_lstm=data_dict["X_train_lstm"],
        y_train_lstm=data_dict["y_train_lstm"],
        X_test_lstm=data_dict["X_test_lstm"],
        y_test_lstm=data_dict["y_test_lstm"]
    )
    print("✅ Preprocessed data saved to preprocessed_gold_data.npz")

    # 3. Inspect results
    print("\nTrain/Test sets (scaled):")
    print("X_train:", data_dict['X_train'].shape)
    print("X_test:", data_dict['X_test'].shape)
    print("y_train:", data_dict['y_train'].shape)
    print("y_test:", data_dict['y_test'].shape)

    print("\nFor LSTM:")
    print("X_train_lstm:", data_dict['X_train_lstm'].shape)
    print("y_train_lstm:", data_dict['y_train_lstm'].shape)
    print("X_test_lstm:", data_dict['X_test_lstm'].shape)
    print("y_test_lstm:", data_dict['y_test_lstm'].shape)

    print("\nFeature columns:", data_dict['feature_names'][:10], "...")
    print("\nOriginal test dates (for plotting back results):")
    print(data_dict['original_test_dates'][:5])

main()

Combined dataset shape: (1258, 11)
Preparing features...
Creating technical indicators...
Creating rolling features...
Creating lag features...
Dataset shape: (1208, 52)
Features: 53
✅ Preprocessed data saved to preprocessed_gold_data.npz

Train/Test sets (scaled):
X_train: (966, 52)
X_test: (242, 52)
y_train: (966,)
y_test: (242,)

For LSTM:
X_train_lstm: (906, 60, 52)
y_train_lstm: (906,)
X_test_lstm: (182, 60, 52)
y_test_lstm: (182,)

Feature columns: ['Date', 'gold_open', 'gold_high', 'gold_low', 'gold_close', 'gold_volume', 'dxy_close', 'sp500_close', 'treasury_close', 'oil_close'] ...

Original test dates (for plotting back results):
Index([1015, 1016, 1017, 1018, 1019], dtype='int64')
